In [2]:
import tensorflow as tf
from tensorflow.keras import layers, models


# ============================================================
# 1. PATHS
# ============================================================

train_path = r"E:\learning\learning dataset\Train"
validation_path = r"E:\learning\learning dataset\validation"


# ============================================================
# 2. BASIC SETTINGS
# ============================================================

IMG_SIZE = (224, 224)
BATCH_SIZE = 32


# ============================================================
# 3. LOAD TRAINING DATASET
# ============================================================

train_dataset = tf.keras.utils.image_dataset_from_directory(
    train_path,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="int",
    shuffle=True
)


# ============================================================
# 4. LOAD VALIDATION DATASET
# ============================================================

validation_dataset = tf.keras.utils.image_dataset_from_directory(
    validation_path,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="int",
    shuffle=False
)


# ============================================================
# 5. CHECK CLASS NAMES
# ============================================================

class_names = train_dataset.class_names



num_classes = len(class_names)




# ============================================================
# 6. DATA AUGMENTATION
# ============================================================

augmentation_model = tf.keras.Sequential([

    layers.RandomFlip("horizontal"),

    layers.RandomRotation(0.1),

    layers.RandomZoom(0.1)

])


# ============================================================
# 7. VGG16 PREPROCESSING
# ============================================================

def preprocess_train(image, label):

    # Apply augmentation ONLY to training images
    image = augmentation_model(image)

    # VGG16 ImageNet preprocessing
    image = tf.keras.applications.vgg16.preprocess_input(image)

    return image, label


def preprocess_validation(image, label):

    # NO augmentation for validation
    image = tf.keras.applications.vgg16.preprocess_input(image)

    return image, label


# ============================================================
# 8. APPLY PREPROCESSING
# ============================================================

train_dataset = train_dataset.map(
    preprocess_train,
    num_parallel_calls=tf.data.AUTOTUNE
)

validation_dataset = validation_dataset.map(
    preprocess_validation,
    num_parallel_calls=tf.data.AUTOTUNE
)


# ============================================================
# 9. PREFETCH
# ============================================================

train_dataset = train_dataset.prefetch(
    tf.data.AUTOTUNE
)

validation_dataset = validation_dataset.prefetch(
    tf.data.AUTOTUNE
)


# ============================================================
# 10. LOAD PRETRAINED VGG16
# ============================================================

vgg16_base = tf.keras.applications.VGG16(

    weights="imagenet",

    include_top=False,

    input_shape=(224, 224, 3)

)


# ============================================================
# 11. FREEZE VGG16
# ============================================================

vgg16_base.trainable = False


# ============================================================
# 12. BUILD CLASSIFICATION MODEL
# ============================================================

model = models.Sequential([

    # Pretrained VGG16
    vgg16_base,

    # Convert feature maps into feature vector
    layers.GlobalAveragePooling2D(),

    # Our classifier
    layers.Dense(
        128,
        activation="relu"
    ),

    # Reduce overfitting
    layers.Dropout(0.5),

    # Output layer
    layers.Dense(
        num_classes,
        activation="softmax"
    )

])



# ============================================================
# 14. COMPILE MODEL
# ============================================================

model.compile(

    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.001
    ),

    loss="sparse_categorical_crossentropy",

    metrics=["accuracy"]

)


# ============================================================
# 15. TRAIN MODEL
# ============================================================

history = model.fit(

    train_dataset,

    validation_data=validation_dataset,

    epochs=10

)


# ============================================================
# 16. FINAL VALIDATION RESULT
# ============================================================

validation_loss, validation_accuracy = model.evaluate(
    validation_dataset
)



print("Validation Loss:",
      validation_loss)

print("Validation Accuracy:",
      validation_accuracy)

Found 720 files belonging to 10 classes.
Found 200 files belonging to 10 classes.
Epoch 1/10
23/23 [==============================] - 420s 18s/step - loss: 5.2820 - accuracy: 0.2458 - val_loss: 1.0133 - val_accuracy: 0.6600
Epoch 2/10
23/23 [==============================] - 416s 18s/step - loss: 1.5465 - accuracy: 0.5125 - val_loss: 0.4698 - val_accuracy: 0.8300
Epoch 3/10
23/23 [==============================] - 340s 15s/step - loss: 1.0533 - accuracy: 0.6278 - val_loss: 0.2716 - val_accuracy: 0.9050
Epoch 4/10
23/23 [==============================] - 288s 13s/step - loss: 0.7886 - accuracy: 0.7097 - val_loss: 0.1341 - val_accuracy: 0.9550
Epoch 5/10
23/23 [==============================] - 361s 16s/step - loss: 0.5813 - accuracy: 0.8056 - val_loss: 0.0734 - val_accuracy: 0.9850
Epoch 6/10
23/23 [==============================] - 429s 19s/step - loss: 0.4853 - accuracy: 0.8306 - val_loss: 0.0499 - val_accuracy: 0.9950
Epoch 7/10
23/23 [==============================] - 399s 17s/step 

In [4]:
# Save the entire CNN model to an H5 file
model.save("vgg16_dog breeds.h5")
